# Professional Practice X2: Dimensionality Reduction Pipeline

## Production-Ready Pipeline for High-Dimensional Data

**Production Challenge**: Your dataset has 1000 features. How do you visualize it, reduce dimensionality, and preserve information?

**Goal**: Build a complete pipeline comparing PCA, t-SNE, and UMAP with proper preprocessing, evaluation, and visualization.

**Use Case**: Exploratory data analysis, feature engineering, visualization

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits, fetch_olivetti_faces
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from sklearn.cluster import KMeans
import umap
import time

np.random.seed(42)
print('✅ Loaded!')

## Step 1: Load and Explore Data

We'll use the digits dataset (64 features, 10 classes).

In [ ]:
# Load data
digits = load_digits()
X = digits.data
y = digits.target

print(f"Dataset: {X.shape}")
print(f"Features: {X.shape[1]}")
print(f"Samples: {X.shape[0]}")
print(f"Classes: {len(np.unique(y))}")
print(f"\nFeature statistics:")
print(f"  Range: [{X.min():.2f}, {X.max():.2f}]")
print(f"  Mean: {X.mean():.2f}")
print(f"  Std: {X.std():.2f}")

# Visualize samples
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(X[i].reshape(8, 8), cmap='gray')
    ax.set_title(f'Label: {y[i]}')
    ax.axis('off')
plt.suptitle('Sample Digits', fontweight='bold')
plt.tight_layout()
plt.show()
print('✅ Data loaded!')

## Step 2: Preprocessing

Standardization is crucial for distance-based methods.

In [ ]:
# Standardize features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f"After standardization:")
print(f"  Mean: {X_scaled.mean():.2e}")
print(f"  Std: {X_scaled.std():.2f}")
print(f"  Range: [{X_scaled.min():.2f}, {X_scaled.max():.2f}]")
print('✅ Data standardized!')

## Step 3: Apply Dimensionality Reduction Methods

Compare PCA, t-SNE, and UMAP on the same data.

In [ ]:
# === PCA ===
print("Running PCA...")
start = time.time()
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pca_time = time.time() - start

print(f"  Time: {pca_time:.3f}s")
print(f"  Explained variance: {pca.explained_variance_ratio_.sum():.2%}")
print(f"  Component 1: {pca.explained_variance_ratio_[0]:.2%}")
print(f"  Component 2: {pca.explained_variance_ratio_[1]:.2%}")

# === t-SNE ===
print("\nRunning t-SNE...")
start = time.time()
tsne = TSNE(n_components=2, random_state=42, perplexity=30, n_iter=1000)
X_tsne = tsne.fit_transform(X_scaled)
tsne_time = time.time() - start
print(f"  Time: {tsne_time:.3f}s")
print(f"  Final KL divergence: {tsne.kl_divergence_:.2f}")

# === UMAP ===
print("\nRunning UMAP...")
start = time.time()
reducer = umap.UMAP(n_components=2, random_state=42, n_neighbors=15, min_dist=0.1)
X_umap = reducer.fit_transform(X_scaled)
umap_time = time.time() - start
print(f"  Time: {umap_time:.3f}s")

print('\n✅ All methods applied!')

## Step 4: Evaluate Cluster Quality

Use silhouette score on K-Means clustering of reduced data.

In [ ]:
# Cluster and evaluate
n_clusters = 10
methods = {
    'PCA': X_pca,
    't-SNE': X_tsne,
    'UMAP': X_umap
}

print("=== CLUSTERING EVALUATION ===\n")
for name, X_reduced in methods.items():
    # K-Means on reduced data
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_reduced)
    
    # Evaluate
    sil = silhouette_score(X_reduced, labels)
    
    print(f"{name}:")
    print(f"  Silhouette: {sil:.3f}")
    
print('\n✅ Evaluation complete!')

## Step 5: Visualize and Compare

In [ ]:
# Comprehensive visualization
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# === Row 1: Colored by true label ===
for idx, (name, X_r) in enumerate(methods.items()):
    ax = axes[0, idx]
    scatter = ax.scatter(X_r[:, 0], X_r[:, 1], c=y, cmap='tab10', 
                        s=20, alpha=0.7, edgecolors='black', linewidths=0.3)
    ax.set_xlabel('Component 1', fontsize=11)
    ax.set_ylabel('Component 2', fontsize=11)
    ax.set_title(f'{name} (colored by digit)', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    
    if idx == 2:
        plt.colorbar(scatter, ax=ax, label='Digit')

# === Row 2: Colored by K-Means cluster ===
for idx, (name, X_r) in enumerate(methods.items()):
    ax = axes[1, idx]
    
    # K-Means
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_r)
    sil = silhouette_score(X_r, labels)
    
    scatter = ax.scatter(X_r[:, 0], X_r[:, 1], c=labels, cmap='tab10',
                        s=20, alpha=0.7, edgecolors='black', linewidths=0.3)
    ax.scatter(kmeans.cluster_centers_[:, 0], kmeans.cluster_centers_[:, 1],
              c='red', s=200, marker='X', edgecolors='black', linewidths=2,
              label='Centroids')
    ax.set_xlabel('Component 1', fontsize=11)
    ax.set_ylabel('Component 2', fontsize=11)
    ax.set_title(f'{name} K-Means (Sil: {sil:.3f})', fontweight='bold', fontsize=12)
    ax.grid(True, alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()
print('✅ Visualization complete!')

## Step 6: PCA Scree Plot

How many components should we keep?

In [ ]:
# Full PCA for scree plot
pca_full = PCA(random_state=42)
pca_full.fit(X_scaled)

# Cumulative explained variance
cumsum = np.cumsum(pca_full.explained_variance_ratio_)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scree plot
ax = axes[0]
ax.plot(range(1, len(pca_full.explained_variance_ratio_) + 1),
       pca_full.explained_variance_ratio_, 'bo-', linewidth=2)
ax.set_xlabel('Principal Component', fontsize=12)
ax.set_ylabel('Explained Variance Ratio', fontsize=12)
ax.set_title('Scree Plot', fontweight='bold', fontsize=13)
ax.grid(True, alpha=0.3)

# Cumulative variance
ax = axes[1]
ax.plot(range(1, len(cumsum) + 1), cumsum, 'ro-', linewidth=2)
ax.axhline(y=0.8, color='green', linestyle='--', linewidth=2, label='80% threshold')
ax.axhline(y=0.95, color='orange', linestyle='--', linewidth=2, label='95% threshold')
ax.set_xlabel('Number of Components', fontsize=12)
ax.set_ylabel('Cumulative Explained Variance', fontsize=12)
ax.set_title('Cumulative Variance Explained', fontweight='bold', fontsize=13)
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Find number of components for 95% variance
n_comp_95 = np.argmax(cumsum >= 0.95) + 1
print(f"Components needed for 95% variance: {n_comp_95}/{len(cumsum)}")
print('✅ Scree plot complete!')

## Step 7: Timing Comparison

In [ ]:
# Compare computation time
times = {
    'PCA': pca_time,
    't-SNE': tsne_time,
    'UMAP': umap_time
}

plt.figure(figsize=(10, 6))
bars = plt.bar(times.keys(), times.values(), color=['steelblue', 'coral', 'seagreen'],
              edgecolor='black', linewidth=1.5, alpha=0.8)
plt.ylabel('Time (seconds)', fontsize=12)
plt.title('Computation Time Comparison', fontweight='bold', fontsize=14)
plt.grid(True, alpha=0.3, axis='y')

# Add value labels
for bar, (name, time_val) in zip(bars, times.items()):
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., height,
            f'{time_val:.2f}s',
            ha='center', va='bottom', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()
print('✅ Timing comparison complete!')

## Production Decision Framework

### Method Comparison

| Method | Speed | Preserves | Best For | Deterministic |
|--------|-------|-----------|----------|--------------|
| **PCA** | ⚡⚡⚡ | Global structure, variance | Preprocessing, noise reduction, visualization | ✅ Yes |
| **t-SNE** | 🐌 | Local structure, clusters | Visualization only | ❌ No (random init) |
| **UMAP** | ⚡⚡ | Both local & global | Visualization, clustering | ✅ Yes (with seed) |

### When to Use Each

**PCA**:
- ✅ Large datasets (millions of samples)
- ✅ Need interpretable components
- ✅ Preprocessing for other algorithms
- ✅ Noise reduction
- ❌ Non-linear patterns

**t-SNE**:
- ✅ Publication-quality visualization
- ✅ Identifying clusters visually
- ✅ Small to medium datasets (< 10k samples)
- ❌ Distance preservation
- ❌ Adding new points

**UMAP**:
- ✅ Best of both worlds (speed + quality)
- ✅ Large datasets
- ✅ Preserves global structure
- ✅ Can transform new data
- ✅ General-purpose reduction

### Production Pipeline Template

```python
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import umap

# 1. Standardize
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. PCA for noise reduction (keep 95% variance)
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

# 3. UMAP for final reduction
reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1)
X_final = reducer.fit_transform(X_pca)

# 4. Use for clustering, visualization, etc.
```

### Production Checklist

✅ **Always standardize** before dimensionality reduction  
✅ **Use PCA first** for high-dimensional data (>100 dims)  
✅ **Choose n_components** to retain 95% variance  
✅ **Validate with domain knowledge** (do clusters make sense?)  
✅ **Monitor computation time** on production data  
✅ **Save fitted transformers** for consistent preprocessing